# M1.2 · Prompting 🎭

**Lección oficial:** *Prompting* · **Notebook original:** `module-1/1.1_prompting.ipynb`

## 🎯 Qué vas a aprender
- Qué es el **system prompt** (las instrucciones del agente).
- **Few-shot**: enseñar con ejemplos.
- **Prompts estructurados**: reglas claras de formato.
- **Structured output**: que el agente responda con datos ordenados (Pydantic).

## 🧸 Explicado sencillo
El **system prompt** es como el **guion de un actor** 🎭: le dice quién es y cómo debe comportarse antes de que empiece la obra. El usuario nunca lo ve, pero cambia todo.

## 🛠️ Paso 0 · Preparar el notebook

Esta celda hace tres cosas por ti:
1. Encuentra la carpeta principal del repo (donde vive `local_model.py`).
2. Lee tu archivo `.env` (tus llaves y la dirección del servidor) **sin mostrarlas**.
3. Revisa el **semáforo** del servidor Ollama, que es **compartido**:
   - 🟢 libre → adelante.
   - 🟡 hay un modelo cargado → si fuiste tú hace un rato, sigue; si no, espera.
   - 🔴 sin conexión → revisa `OLLAMA_BASE_URL` en tu `.env`.

> Corre las celdas **en orden, de arriba hacia abajo** (Shift + Enter). El número `[n]` a la izquierda te dice en qué orden se ejecutaron.

In [ ]:
# 🟢 ADAPTADO LOCAL: preparar entorno (ver ayuda.py)
from ayuda import preparar, ver, semaforo
preparar()

from local_model import get_model   # se importa DESPUÉS de cargar el .env

## 1 · Sin instrucciones
Primero, un agente sin guion. Fíjate en lo que responde.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre
# agent = create_agent(model="gpt-5-nano")

# 🟢 ADAPTADO LOCAL
from langchain.agents import create_agent
from langchain.messages import HumanMessage

question = HumanMessage(content="What's the capital of the moon?")

agent = create_agent(model=get_model())
response = agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

## 2 · Con system prompt 🎭
Ahora le damos un **personaje**: escritor de ciencia ficción. Misma pregunta, otra respuesta.

In [ ]:
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(model=get_model(), system_prompt=system_prompt)
response = scifi_agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

## 3 · Few-shot: enseñar con ejemplos 📚
*Few-shot* = "pocos ejemplos". En vez de explicar con palabras qué quieres, **muestras** 2 o 3 ejemplos y el modelo imita el patrón.

🧸 Es como enseñarle a alguien un juego: "si digo Marte, tú dices Marsialis; si digo Venus, tú dices Venusovia… ¿y la Luna?"

In [ ]:
system_prompt = """
You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia
"""

scifi_agent = create_agent(model=get_model(), system_prompt=system_prompt)
response = scifi_agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

🤔 **¿Respondió solo un nombre, o escribió un párrafo?** Los modelos pequeños a veces "se emocionan" y escriben de más. Lo arreglamos con reglas claras.

## 4 · Prompt estructurado: reglas claras ✅
Le decimos **exactamente** el formato: solo el nombre, una o dos palabras, sin explicación.

🧠 **Mnemotecnia R-E-F:** un buen prompt tiene **R**ol (quién eres), **E**jemplos (cómo se ve) y **F**ormato (cómo respondes).

In [ ]:
system_prompt = """
You are a science fiction writer. Invent a capital city name for the place the user asks about.
Respond ONLY with the city name. One or two words. No explanation, no markdown.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia
"""

scifi_agent = create_agent(model=get_model(), system_prompt=system_prompt)
response = scifi_agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

### 🎁 Extra: ejemplos como mensajes
Otra forma de hacer few-shot: en vez de escribir los ejemplos dentro del prompt, se pasan como **mensajes reales** (Humano → Modelo). A los modelos pequeños esto les funciona muy bien, porque "creen" que ya respondieron así antes.

In [ ]:
from langchain.messages import AIMessage

scifi_agent = create_agent(
    model=get_model(),
    system_prompt="You are a science fiction writer. Reply ONLY with an invented capital city name.",
)
response = scifi_agent.invoke({"messages": [
    HumanMessage("What is the capital of Mars?"),  AIMessage("Marsialis"),
    HumanMessage("What is the capital of Venus?"), AIMessage("Venusovia"),
    question,
]})
print(response["messages"][-1].content)

## 5 · Structured output: respuestas como datos 🗂️
A veces no queremos un texto bonito, sino **datos** que el programa pueda usar: nombre, ubicación, ambiente, economía.

Para eso definimos un **molde** con `pydantic.BaseModel`, y el agente rellena cada campo.

🧸 Es como darle a alguien un **formulario** en vez de una hoja en blanco.

⚠️ El modelo necesita saber usar *tools* para esto (`qwen3:8b` sí sabe; `gemma3:4b` no).

In [ ]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=get_model(),
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo,
)

response = agent.invoke({"messages": [HumanMessage(content="What is the capital of The Moon?")]})
capital = response["structured_response"]
capital

In [ ]:
# Ahora cada dato se usa como una variable normal de Python
print(f"{capital.name} is a city located at {capital.location}")
print("Ambiente:", capital.vibe)

## ✍️ Tu turno
1. Cambia el personaje del system prompt: por ejemplo, *"Eres una pirata que responde en español"*.
2. Haz un few-shot con **tus propios** ejemplos (por ejemplo, nombres de dragones por color).
3. Crea tu propio molde `BaseModel`, por ejemplo `Mascota` con `nombre`, `especie` y `superpoder`, y pídele al agente que invente una.

## ✅ Mini quiz
1. ¿Qué es el system prompt?
2. ¿Qué significa *few-shot*?
3. ¿Dónde encuentras la respuesta cuando usas `response_format`?
4. ¿Qué tres partes tiene un buen prompt según R-E-F?

<details><summary>Respuestas</summary>

1. Las instrucciones fijas del agente: su "guion".
2. Enseñar con pocos ejemplos.
3. En `response["structured_response"]`.
4. Rol, Ejemplos y Formato.
</details>

## 🧠 Resumen en una línea
**System prompt = guion del actor. Few-shot = enseñar con ejemplos. Structured output = formulario en vez de hoja en blanco.**